In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from src.gaussian_search import qcels
import json

rng = np.random.default_rng()
target = 1/3.0
length = 13

In [ ]:
for i in range(11, length):
    with open(f'./output/invariance/{2**(2*i+2)}_a.jsonl', 'x') as f:
        for _ in range(1):
            qcls_diff = [qcels(target, j, 2**(2*i+2)//j, rng, sigma = 3, sample_repeat=100, G=2**(i+2)//j) for j in 2**np.arange(np.max((1, i-7)), i)]
            y, x, z = zip(*qcls_diff)
            json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
            f.write("\n")

In [ ]:
plt.rcParams.update({'font.size': 10})
plt.rcParams.update({'font.family': "serif"})

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for i in range(2, 13):
    with open(f'./output/invariance/{2**(2*i+2)}.jsonl', 'r') as json_file:
        json_list = list(json_file)
    query = []
    depth = []
    value = []
    for json_str in json_list:
        res = json.loads(json_str)
        query.append(res["query"])
        depth.append(res["depth"])
        value.append(res["value"])
    query = np.max(query, axis= 0)
    depth = np.max(depth, axis= 0)
    error = np.mean(np.abs(np.stack(value, axis=0)-target), axis=0)
    ax.bar3d(np.log2(query), np.log2(depth), np.zeros(len(query)), np.ones(len(query)), np.ones(len(query)), np.log10(error)+4.5, label=f"$DV=2^{{{2*i+2}}}$")
ax.set_xlabel("Volume")
ax.set_ylabel("Depth")
ax.set_zlabel("Error")
ax.set_xticks(range(4, 2 * length, 2), [f"$2^{{{j}}}$" for j in range(4, 2 * length, 2)])
ax.set_yticks(range(2, length, 2), [f"$2^{{{j}}}$" for j in range(2, length, 2)])
ax.set_zticks(np.arange(0.5, 4.5), [f"$10^{{{int(j-4.5)}}}$" for j in np.arange(0.5, 4.5)])
ax.legend()
ax.invert_yaxis()
plt.savefig("invariance.pdf")